# GINN v2: more data, smaller network, regularisation, VIX
Mkt-RF daily, 1990 to Aug 2026. Splits by date, never shuffled:
train 1990-2014 (fit everything), validation 2015-2019 (early stopping and settings), test 2020-2026.

Baselines fitted on train only, then frozen: GJR and GJR-X (GJR plus δ·VIX²). Two networks on top of GJR:
GINN (inputs r, |r|, log h_GJR) and GINN-VIX (same plus log VIX²).
Questions: does GINN beat GJR? Does GINN-VIX beat GJR-X, i.e. add something beyond VIX itself?

**Run 2 changes** (run 1 is kept in `old/ginn_v2_run1/`):
1. The GJR / GJR-X fits use one FFT convolution instead of a Python loop (same numbers, about 1000x faster)
   and L-BFGS instead of Adam, so GJR-X now converges. In run 1 it stopped at 0.38835 instead of 0.38721.
2. New section 7b: the same networks trained with ex-ante volatility weights (GINN-W, GINN-VIX-W).
   Settings are fixed in advance, not tuned. This is a second look at the test period, so it is reported as a robustness check.

In [4]:
import copy
import time
from pathlib import Path
import numpy as np
import pandas as pd
import torch
import torch.nn as nn

nnf = torch.nn.functional
CANDIDATES = (Path('../data'), Path('data'), Path('..'), Path('.'))   # after the tidy: ../data; before it: the project root
DATA = next((p for p in CANDIDATES if (p / 'VIXCLS.csv').exists()), None)
assert DATA is not None, f'VIXCLS.csv not found from {Path.cwd()}'
OUT = Path('outputs')
print('data folder:', DATA.resolve())

data folder: D:\Hanyang_4(2)\Introduction to Financial Engineering\fe_project


## 1. Data: returns and VIX, aligned by date
VIX is the close of each day; a feature for day t only ever uses days up to t-1. The few market days with no VIX are forward-filled.

In [5]:
d = pd.read_parquet(DATA / 'ff_daily_garchdata_long.parquet')
d = d[d['series'] == 'Mkt-RF'].copy()
date_col = 'Date' if 'Date' in d.columns else 'date'      # the date column, whatever its exact name
try:
    d['date'] = pd.to_datetime(d[date_col], format='%d-%m-%Y')   # stored as text like 01-07-1963
except (ValueError, TypeError):
    d['date'] = pd.to_datetime(d[date_col])                       # already a date, or ISO text

vix = pd.read_csv(DATA / 'VIXCLS.csv')
vix.columns = ['date', 'vix']
vix['date'] = pd.to_datetime(vix['date'])
vix['vix'] = pd.to_numeric(vix['vix'], errors='coerce')

m = d[['date', 'ret']].merge(vix, on='date', how='left')
m = m[m['date'] >= '1990-01-02'].sort_values('date').reset_index(drop=True)
print('market days without a VIX close (forward-filled):', int(m['vix'].isna().sum()))
m['vix'] = m['vix'].ffill()

dates = m['date'].to_numpy()
r_all = torch.tensor(m['ret'].to_numpy() * 100, dtype=torch.float64)          # percent
x_all = torch.tensor(m['vix'].to_numpy() ** 2 / 252, dtype=torch.float64)     # VIX^2 as a daily variance, %^2

TRAIN_END, VAL_END = np.datetime64('2015-01-01'), np.datetime64('2020-01-01')
N_TRAIN = int((dates < TRAIN_END).sum())
r_train, x_train = r_all[:N_TRAIN], x_all[:N_TRAIN]
print(f"{str(dates[0])[:10]} to {str(dates[-1])[:10]} | {len(r_all)} days | train days {N_TRAIN} (expected 9233 and 6301)")

market days without a VIX close (forward-filled): 3
1990-01-02 to 2026-08-31 | 9233 days | train days 6301 (expected 9233 and 6301)


## 2. GJR and GJR-X in torch (one function for both)
h(t) = ω + (α + γ·1{r(t−1)<0})·r²(t−1) + β·h(t−1) + δ·VIX²(t−1).   GJR is the special case δ = 0.
raw has 4 numbers for GJR, 5 for GJR-X. δ = softplus(raw[4])·0.1 ≥ 0. Softmax over [α, γ/2, β, leftover] as before.

**Why a convolution.** Write c(t−1) = ω + (α + γ·1{r(t−1)<0})·r²(t−1) + δ·x(t−1). Then h(t) = c(t−1) + β·h(t−1).
Unroll it: h(t) = c(t−1) + β·c(t−2) + β²·c(t−3) + … + β^t·h(0). That is the series c convolved with the kernel 1, β, β², …
`torch.fft` computes a convolution of 6,000 numbers in one step, and autograd differentiates through it,
so the 6,000-step Python loop (about 2.5 s per evaluation) becomes a few milliseconds. The loop version stays below as a check.

**Why L-BFGS.** With 4 or 5 parameters and a smooth full-batch loss, a quasi-Newton method builds a curvature estimate
and takes well-scaled steps. Adam takes small steps of fixed size and crawls along flat valleys, which is why GJR-X had not converged after 400 steps.

In [6]:
def unpack_gjr(raw):
    omega = nnf.softplus(raw[0]) * 0.1
    w = torch.softmax(torch.cat([raw[1:4], torch.zeros(1, dtype=raw.dtype)]), dim=0)   # [alpha, gamma/2, beta, leftover]
    delta = nnf.softplus(raw[4]) * 0.1 if len(raw) > 4 else torch.zeros((), dtype=raw.dtype)
    return omega, w[0], 2 * w[1], w[2], delta


def gjr_filter(raw, r, x, h0):
    # variance path h(0..T-1) as one convolution: h(t) = sum_s beta^(t-1-s) c(s) + beta^t h(0)
    omega, alpha, gamma, beta, delta = unpack_gjr(raw)
    T = len(r)
    neg = (r[:-1] < 0).to(r.dtype)
    c = omega + (alpha + gamma * neg) * r[:-1] ** 2 + delta * x[:-1]       # c[s] feeds h[s+1]
    j = torch.arange(T - 1, dtype=r.dtype)
    kernel = beta ** j                                                      # 1, beta, beta^2, ...
    n = 2 * T                                                               # zero padding, so the FFT does not wrap around
    conv = torch.fft.irfft(torch.fft.rfft(c, n=n) * torch.fft.rfft(kernel, n=n), n=n)[:T - 1]
    return torch.cat([h0.reshape(1), conv + h0 * beta ** (j + 1)])


def gjr_path_loop(raw, r, x, h0):
    # the plain recursion, kept only to check gjr_filter
    omega, alpha, gamma, beta, delta = unpack_gjr(raw)
    h = torch.empty_like(r)
    h[0] = h0
    for t in range(1, len(r)):
        neg = (r[t - 1] < 0).to(r.dtype)
        h[t] = omega + (alpha + gamma * neg) * r[t - 1] ** 2 + beta * h[t - 1] + delta * x[t - 1]
    return h


def nll_gjr(raw, r, x):
    h = gjr_filter(raw, r, x, r.var())
    return 0.5 * torch.mean(torch.log(h[1:]) + r[1:] ** 2 / h[1:])


def fit(raw0, r, x, max_iter=500):
    raw = raw0.clone().requires_grad_(True)
    opt = torch.optim.LBFGS([raw], lr=1.0, max_iter=max_iter, history_size=20,
                            tolerance_grad=1e-10, tolerance_change=1e-14, line_search_fn='strong_wolfe')

    def closure():                      # L-BFGS re-evaluates the loss several times per step, so it needs a function
        opt.zero_grad()
        loss = nll_gjr(raw, r, x)
        loss.backward()
        return loss

    t0 = time.time()
    opt.step(closure)                   # one call runs up to max_iter iterations
    n_iter = opt.state[raw]['n_iter']
    raw = raw.detach()
    g = raw.clone().requires_grad_(True)
    nll_gjr(g, r, x).backward()
    loss = nll_gjr(raw, r, x).item()
    print(f'L-BFGS {n_iter} iterations, {time.time() - t0:.1f}s | loss {loss:.5f} | largest |gradient| {g.grad.abs().max().item():.1e}')
    return raw, loss


with torch.no_grad():                   # check: the convolution reproduces the loop
    test_raw = torch.tensor([-1.0, 0.405, 0.916, 3.807, 0.0], dtype=torch.float64)
    h_fft = gjr_filter(test_raw, r_all, x_all, r_train.var())
    h_loop = gjr_path_loop(test_raw, r_all, x_all, r_train.var())
print('max relative difference, convolution vs loop:', f'{(h_fft / h_loop - 1).abs().max().item():.1e}', '(should be below 1e-10)')

max relative difference, convolution vs loop: 7.5e-15 (should be below 1e-10)


In [7]:
raw_gjr, loss_gjr = fit(torch.tensor([-1.0, 0.405, 0.916, 3.807], dtype=torch.float64), r_train, x_train)
print('GJR   omega, alpha, gamma, beta =', [round(p.item(), 4) for p in unpack_gjr(raw_gjr)[:4]], '| loss', round(loss_gjr, 5))
print('target (scipy):                 [0.0167, 0.0036, 0.1426, 0.9105] | loss 0.39807')

L-BFGS 20 iterations, 0.6s | loss 0.39807 | largest |gradient| 4.4e-09
GJR   omega, alpha, gamma, beta = [0.0167, 0.0036, 0.1426, 0.9105] | loss 0.39807
target (scipy):                 [0.0167, 0.0036, 0.1426, 0.9105] | loss 0.39807


In [8]:
raw_gjrx, loss_gjrx = fit(torch.cat([raw_gjr, torch.zeros(1, dtype=torch.float64)]), r_train, x_train)
print('GJR-X omega, alpha, gamma, beta, delta =', [round(p.item(), 4) for p in unpack_gjr(raw_gjrx)], '| loss', round(loss_gjrx, 5))
print('target (scipy):                        [0.0042, 0.0, 0.1797, 0.8047, 0.0642] | loss 0.38721')
print('alpha sits at its lower bound 0, so torch reports a tiny positive alpha; the loss is what must match')

L-BFGS 93 iterations, 3.3s | loss 0.38721 | largest |gradient| 8.8e-09
GJR-X omega, alpha, gamma, beta, delta = [0.0042, 0.0, 0.1797, 0.8046, 0.0643] | loss 0.38721
target (scipy):                        [0.0042, 0.0, 0.1797, 0.8047, 0.0642] | loss 0.38721
alpha sits at its lower bound 0, so torch reports a tiny positive alpha; the loss is what must match


## 3. Frozen variance paths for every day

In [9]:
with torch.no_grad():
    h_gjr = gjr_filter(raw_gjr, r_all, x_all, r_train.var())
    h_gjrx = gjr_filter(raw_gjrx, r_all, x_all, r_train.var())
for name, h, ref in (('GJR', h_gjr, loss_gjr), ('GJR-X', h_gjrx, loss_gjrx)):
    chk = 0.5 * torch.mean(torch.log(h[1:N_TRAIN]) + r_all[1:N_TRAIN] ** 2 / h[1:N_TRAIN])
    print(f'{name}: loss from path on train days {chk.item():.5f} | fitted {ref:.5f}')

GJR: loss from path on train days 0.39807 | fitted 0.39807
GJR-X: loss from path on train days 0.38721 | fitted 0.38721


## 4. Features, windows and loss weights
Base features: r, |r|, log h_GJR. VIX features: the same plus log VIX². Standardised with train-day statistics.
Target day t uses days t-22 … t-1. Windows are assigned to train / val / test by the target's date.

**Weights for section 7b:** w(t) = sqrt(h_GJR(t) / mean train h_GJR), rescaled to mean 1 on train days.
- **Ex-ante:** h_GJR(t) is known at the end of day t−1, so the weight never looks at the outcome.
  With such weights the true variance still minimises the expected loss on every day, so the forecast target is unchanged;
  only which days a misspecified model fits best changes. Weighting by the realised r²(t) would push the forecasts up (biased).
- **Square root:** weighting by variance itself would give 2008 alone 18% of the total weight (effective sample about 1,800 days of 6,279).
  Weighting by volatility keeps the effective sample at about 4,900 days.

In [10]:
L = 22
feat_base = torch.stack([r_all, r_all.abs(), torch.log(h_gjr)], dim=1)          # (T, 3)
feat_vix = torch.cat([feat_base, torch.log(x_all).unsqueeze(1)], dim=1)          # (T, 4)


def standardise(Fr):
    mu, sd = Fr[:N_TRAIN].mean(dim=0), Fr[:N_TRAIN].std(dim=0)
    return ((Fr - mu) / sd).to(torch.float32)


Fb, Fv = standardise(feat_base), standardise(feat_vix)
targets = np.arange(L, len(r_all))
tdates = dates[targets]
split = np.where(tdates < TRAIN_END, 'train', np.where(tdates < VAL_END, 'val', 'test'))

Xb = torch.stack([Fb[t - L:t] for t in targets])        # (N, 22, 3)
Xv = torch.stack([Fv[t - L:t] for t in targets])        # (N, 22, 4)
y = r_all[targets].to(torch.float32)
hb = h_gjr[targets].to(torch.float32)                    # base variance for both networks
hx = h_gjrx[targets].to(torch.float32)                   # GJR-X, a baseline only

for i in (0, len(targets) // 2, len(targets) - 1):
    t = targets[i]
    assert torch.equal(Xv[i, -1], Fv[t - 1]) and torch.equal(Xv[i, 0], Fv[t - L])

masks = {s: torch.tensor(split == s) for s in ('train', 'val', 'test')}
print('Xb', tuple(Xb.shape), '| Xv', tuple(Xv.shape), '|', {s: int(mk.sum()) for s, mk in masks.items()},
      '(expected train 6279, val 1258, test 1674)')

w_vol = torch.sqrt(hb / hb[masks['train']].mean())
w_vol = w_vol / w_vol[masks['train']].mean()
wt = w_vol[masks['train']]
print(f'weights on train days: max {wt.max().item():.1f} | effective sample {(wt.sum() ** 2 / (wt ** 2).sum()).item():.0f} of {len(wt)} days')

Xb (9211, 22, 3) | Xv (9211, 22, 4) | {'train': 6279, 'val': 1258, 'test': 1674} (expected train 6279, val 1258, test 1674)
weights on train days: max 5.3 | effective sample 4880 of 6279 days


## 5. Model and training function
Same GINN as v1, with a choice of hidden size and weight decay (L2 regularisation inside Adam), and optional loss weights.
Early stopping on validation, using the same (weighted or unweighted) loss as training.
The starting point (epoch 0) is plain GJR, so a network that never beats GJR on validation returns GJR.

In [11]:
class GINN(nn.Module):
    def __init__(self, n_features, hidden):
        super().__init__()
        self.lstm = nn.LSTM(input_size=n_features, hidden_size=hidden, batch_first=True)
        self.head = nn.Linear(hidden, 1)
        nn.init.zeros_(self.head.weight)
        nn.init.zeros_(self.head.bias)

    def forward(self, x, h_base):
        out, _ = self.lstm(x)
        g = self.head(out[:, -1, :]).squeeze(-1)
        return h_base * torch.exp(g)


def qmle(h, r, w=None):
    per_day = torch.log(h) + r ** 2 / h
    return 0.5 * (per_day.mean() if w is None else (w * per_day).mean())


def train_ginn(X, base, hidden, wd, seed, w=None, max_epochs=3000, patience=300, lr=1e-3):
    tr, va = masks['train'], masks['val']
    Xtr, btr, ytr, Xva, bva, yva = X[tr], base[tr], y[tr], X[va], base[va], y[va]
    wtr, wva = (None, None) if w is None else (w[tr], w[va])
    torch.manual_seed(seed)
    model = GINN(X.shape[2], hidden)
    opt = torch.optim.Adam(model.parameters(), lr=lr, weight_decay=wd)
    best_val, best_state, best_epoch = qmle(bva, yva, wva).item(), copy.deepcopy(model.state_dict()), 0
    for epoch in range(1, max_epochs + 1):
        model.train()
        opt.zero_grad()
        loss = qmle(model(Xtr, btr), ytr, wtr)
        loss.backward()
        torch.nn.utils.clip_grad_norm_(model.parameters(), max_norm=1.0)
        opt.step()
        if epoch % 10 == 0:
            model.eval()
            with torch.no_grad():
                val = qmle(model(Xva, bva), yva, wva).item()
            if val < best_val:
                best_val, best_state, best_epoch = val, copy.deepcopy(model.state_dict()), epoch
            if epoch - best_epoch >= patience:
                break
    model.load_state_dict(best_state)
    return model, best_val, best_epoch


def ensemble(name, X, hidden, wd, w=None, seeds=(0, 1, 2)):
    # train one model per seed and average the log corrections g: h = h_GJR * exp(mean g)
    gs = []
    for seed in seeds:
        mdl, v, e = train_ginn(X, hb, hidden, wd, seed, w=w)
        mdl.eval()
        with torch.no_grad():
            gs.append(torch.log(mdl(X, hb) / hb))
        print(f'{name:10s} seed {seed} | val {v:.5f} | best epoch {e}')
    return hb * torch.exp(torch.stack(gs).mean(dim=0))

## 6. Small tuning grid, judged on validation only
hidden {8, 16} × weight decay {0, 1e-4, 1e-3}, for each network, seed 0.
Run 1 showed that seed-to-seed differences are about as large as the differences between settings, so read the choice loosely.

In [12]:
print(f"validation loss  GJR {qmle(hb[masks['val']], y[masks['val']]).item():.5f} | GJR-X {qmle(hx[masks['val']], y[masks['val']]).item():.5f}")
grid = []
for name, X in (('GINN', Xb), ('GINN-VIX', Xv)):
    for hidden in (8, 16):
        for wd in (0.0, 1e-4, 1e-3):
            t0 = time.time()
            _, v, e = train_ginn(X, hb, hidden, wd, seed=0)
            grid.append({'model': name, 'hidden': hidden, 'weight_decay': wd, 'val_loss': round(v, 5),
                         'best_epoch': e, 'seconds': round(time.time() - t0)})
            print(grid[-1])
G = pd.DataFrame(grid)
best_cfg = G.loc[G.groupby('model')['val_loss'].idxmin()]
cfg = {row['model']: (int(row['hidden']), float(row['weight_decay'])) for _, row in best_cfg.iterrows()}
print('\nchosen settings:')
print(best_cfg.to_string(index=False))

validation loss  GJR 0.21872 | GJR-X 0.18380
{'model': 'GINN', 'hidden': 8, 'weight_decay': 0.0, 'val_loss': 0.20228, 'best_epoch': 210, 'seconds': 43}
{'model': 'GINN', 'hidden': 8, 'weight_decay': 0.0001, 'val_loss': 0.20277, 'best_epoch': 240, 'seconds': 45}
{'model': 'GINN', 'hidden': 8, 'weight_decay': 0.001, 'val_loss': 0.20236, 'best_epoch': 630, 'seconds': 74}
{'model': 'GINN', 'hidden': 16, 'weight_decay': 0.0, 'val_loss': 0.19712, 'best_epoch': 180, 'seconds': 50}
{'model': 'GINN', 'hidden': 16, 'weight_decay': 0.0001, 'val_loss': 0.2014, 'best_epoch': 240, 'seconds': 57}
{'model': 'GINN', 'hidden': 16, 'weight_decay': 0.001, 'val_loss': 0.20517, 'best_epoch': 440, 'seconds': 78}
{'model': 'GINN-VIX', 'hidden': 8, 'weight_decay': 0.0, 'val_loss': 0.17212, 'best_epoch': 200, 'seconds': 50}
{'model': 'GINN-VIX', 'hidden': 8, 'weight_decay': 0.0001, 'val_loss': 0.17212, 'best_epoch': 210, 'seconds': 51}
{'model': 'GINN-VIX', 'hidden': 8, 'weight_decay': 0.001, 'val_loss': 0.1703

## 7a. Final models: chosen settings, 3 seeds, forecasts averaged

In [13]:
final = {'GINN': ensemble('GINN', Xb, *cfg['GINN']),'GINN-VIX': ensemble('GINN-VIX', Xv, *cfg['GINN-VIX'])}

GINN       seed 0 | val 0.19712 | best epoch 180
GINN       seed 1 | val 0.20367 | best epoch 180
GINN       seed 2 | val 0.20457 | best epoch 130
GINN-VIX   seed 0 | val 0.16641 | best epoch 540
GINN-VIX   seed 1 | val 0.16886 | best epoch 280
GINN-VIX   seed 2 | val 0.16795 | best epoch 360


## 7b. Weighted training (robustness check)
Same settings as section 7 (no new tuning), same seeds, trained and early-stopped on the volatility-weighted loss.
Expect this to change the fit on days that are already turbulent. It cannot fix days like February 2020,
which looked calm the day before; no ex-ante weight can single those out.

In [14]:
final['GINN-W'] = ensemble('GINN-W', Xb, *cfg['GINN'], w=w_vol)
final['GINN-VIX-W'] = ensemble('GINN-VIX-W', Xv, *cfg['GINN-VIX'], w=w_vol)

GINN-W     seed 0 | val 0.28920 | best epoch 200
GINN-W     seed 1 | val 0.28939 | best epoch 170
GINN-W     seed 2 | val 0.29119 | best epoch 170
GINN-VIX-W seed 0 | val 0.26365 | best epoch 340
GINN-VIX-W seed 1 | val 0.26587 | best epoch 300
GINN-VIX-W seed 2 | val 0.26466 | best epoch 300


## 8. Results
- **Unweighted QMLE:** the common yardstick for every model.
- **Weighted QMLE:** shown too, because it is what the -W models were trained on.
- **DM tests on test days:** Newey-West with 22 lags (one trading month); negative means the first model is better.

In [15]:
preds = {'GJR': hb, 'GJR-X (VIX)': hx, **final}


def dm_tstat(dd, lags=22):
    n, u = len(dd), dd - dd.mean()
    lrv = np.mean(u * u) + 2 * sum((1 - k / (lags + 1)) * np.mean(u[k:] * u[:-k]) for k in range(1, lags + 1))
    return dd.mean() / np.sqrt(lrv / n)


wv = w_vol.double().numpy()
L_day = {k: (0.5 * (torch.log(h) + y ** 2 / h)).double().numpy() for k, h in preds.items()}
te = masks['test'].numpy()
table = pd.DataFrame({s: {k: L_day[k][masks[s].numpy()].mean() for k in preds} for s in ('train', 'val', 'test')})
table['test, weighted'] = {k: (wv * L_day[k])[te].mean() for k in preds}
print('mean QMLE loss per day (lower is better)')
print(table.round(5).to_string())

print('\nTest-period comparisons            unweighted loss        weighted loss')
for a, b in (('GINN', 'GJR'), ('GINN-VIX', 'GJR-X (VIX)'), ('GJR-X (VIX)', 'GJR'), ('GINN-VIX', 'GINN'),
             ('GINN-W', 'GJR'), ('GINN-VIX-W', 'GJR-X (VIX)'), ('GINN-W', 'GINN'), ('GINN-VIX-W', 'GINN-VIX')):
    dd, dw = L_day[a][te] - L_day[b][te], (wv * (L_day[a] - L_day[b]))[te]
    print(f'  {a:11s} - {b:12s}: {dd.mean():+.5f} (t {dm_tstat(dd):+.2f})   {dw.mean():+.5f} (t {dm_tstat(dw):+.2f})')

years = pd.DatetimeIndex(tdates).year.to_numpy()
print('\nTest years, mean unweighted loss')
print(pd.DataFrame({yr: {k: L_day[k][te & (years == yr)].mean() for k in preds} for yr in range(2020, 2027)}).round(4).to_string())

mean QMLE loss per day (lower is better)
               train      val     test  test, weighted
GJR          0.39750  0.21872  0.53211         0.83221
GJR-X (VIX)  0.38657  0.18380  0.51365         0.81303
GINN         0.37530  0.19944  0.52305         0.83087
GINN-VIX     0.36859  0.16678  0.50510         0.80929
GINN-W       0.37698  0.20260  0.52979         0.84126
GINN-VIX-W   0.36847  0.16796  0.50325         0.80600

Test-period comparisons            unweighted loss        weighted loss
  GINN        - GJR         : -0.00905 (t -1.20)   -0.00134 (t -0.08)
  GINN-VIX    - GJR-X (VIX) : -0.00855 (t -1.36)   -0.00374 (t -0.39)
  GJR-X (VIX) - GJR         : -0.01846 (t -2.17)   -0.01918 (t -2.70)
  GINN-VIX    - GINN        : -0.01796 (t -2.02)   -0.02158 (t -2.05)
  GINN-W      - GJR         : -0.00232 (t -0.27)   +0.00905 (t +0.44)
  GINN-VIX-W  - GJR-X (VIX) : -0.01040 (t -1.94)   -0.00703 (t -0.66)
  GINN-W      - GINN        : +0.00674 (t +2.15)   +0.01038 (t +1.86)
  GINN-VIX-

In [16]:
OUT.mkdir(exist_ok=True)
out = pd.DataFrame({'date': tdates, 'split': split, 'r': y.numpy(), 'w_vol': w_vol.numpy(),
                    **{k: v.numpy() for k, v in preds.items()}})
out.to_csv(OUT / 'ginn_v2_daily.csv', index=False)
G.to_csv(OUT / 'ginn_v2_grid.csv', index=False)
print(f'saved {OUT / "ginn_v2_daily.csv"} and {OUT / "ginn_v2_grid.csv"} |', out.shape)

saved outputs\ginn_v2_daily.csv and outputs\ginn_v2_grid.csv | (9211, 10)
